# Pipeline Check -- dry run of the full GridRoute + MazeBench pipeline (CHECK/TEST notebook)

Companion to `kaggle_train.ipynb`. Runs the **exact same pipeline, same code, same stages, in the same order** -- setup, feasibility check, Phase 1 baselines, Phase 1b full-precision ablation, Phase 2 recipes 1-3 (SFT -> GRPO timing test -> GRPO full run -> eval, x3 conditions), run summary, results aggregation -- just at a tiny scale (2-16 samples/tasks per stage instead of 20-400, 2-3 GRPO steps instead of 20-hundreds) so the whole thing finishes in well under two hours instead of potentially days.

**What this is for:** catching syntax errors, API-incompatibility (TRL/PEFT version drift), OOM, and memory-not-actually-released-between-stages problems *before* committing real GPU-hours to `kaggle_train.ipynb`. It is not a substitute for that notebook -- 2 samples tell you nothing about accuracy, only whether the code runs. Every stage here calls the real, unmodified project scripts (`eval.py`, `train_sft.py`, `train_grpo.py`, `check_finetune_feasibility.py`) with smaller `--n`/`--n_tasks`/`--max_steps` arguments -- nothing is mocked, stubbed, or faked; every model load, generation, training step, and score below is real.

**How to read the result:** scroll to the **"Pipeline check verdict"** cell at the end (or just check whether this Kaggle run committed with a green checkmark or a red error). Unlike the real notebook -- which deliberately never lets one bad stage kill an unattended multi-hour run -- this notebook raises a real exception and fails the whole commit if *any* stage did not complete cleanly, since "did everything work" is this notebook's entire purpose. If it fails, the failed stage(s) are named explicitly in that cell's output, and the actual error is in that stage's own cell output further up.

**Before running:** identical requirements to `kaggle_train.ipynb` -- GPU T4 x2 (or P100), Internet On, a `GITHUB_TOKEN` secret (fine-grained, read-only, toggled ON for this notebook) to clone the private repo, optionally `HF_TOKEN` for faster downloads. See that notebook's intro cell for the full setup walkthrough; not repeated here.

**Expected runtime:** roughly 45-90 minutes on a Kaggle T4, dominated by model download/load time (4 models loaded repeatedly across stages) more than actual compute -- an estimate, not a guarantee. Still an order of magnitude or more less than the real notebook.

Generated from the same `build_notebook.py` as `kaggle_train.ipynb` (`build_cells(check=True)` vs `build_cells(check=False)`) so the two can't structurally drift apart -- if you change the real pipeline's stages, regenerate both, don't hand-edit either `.ipynb` directly.

In [ ]:
import os, subprocess, sys

REPO_URL = "github.com/Vedang-P/neuro-symbolic-pathfinding.git"
REPO_DIR = "/kaggle/working/neuro-symbolic-pathfinding"

# GIT_TERMINAL_PROMPT=0: if auth still fails for some other reason (token
# lacks access to this repo, expired, etc.), git fails immediately with a
# clear error instead of hanging on an interactive password prompt that has
# nowhere to go in a notebook environment.
env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("Repo already present, pulling latest...")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True, env=env)
elif not os.path.isdir(REPO_DIR):
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        # Token must go in the PASSWORD slot (after the colon), not as a bare
        # username before @ -- a bare-username URL leaves git still needing a
        # password, which it then can't prompt for here ("could not read
        # Password ... No such device or address"). "x-access-token" as the
        # username is GitHub's documented convention for token-based HTTPS auth.
        clone_url = f"https://x-access-token:{token}@{REPO_URL}"
        subprocess.run(["git", "clone", clone_url, REPO_DIR], check=True, env=env)
        print("Cloned via GITHUB_TOKEN secret.")
    except Exception as e:
        print(f"Could not clone via secret ({type(e).__name__}: {e}).")
        print("Check: the secret is attached to this notebook (Add-ons -> Secrets -> make sure")
        print("it's toggled ON for this notebook specifically), the token hasn't expired, and it")
        print("has read access to this exact repo (fine-grained tokens need per-repo access grants).")
        print("If you attached the repo as a Kaggle Dataset instead, set REPO_DIR above to its mount")
        print("path (typically /kaggle/input/<dataset-name>) and re-run this cell.")
        raise

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print("Working dir:", os.getcwd())


In [ ]:
# Pulls in alphamaze_reference/ (github.com/menloresearch/visual-thinker) -- eval.py uses their
# real MazeBench scoring code directly from this submodule when present, falling back to a less
# faithful exact-match approach (with a loud warning) if it isn't.
subprocess.run(["git", "submodule", "update", "--init"], check=True)


In [ ]:
# Optional: read from Kaggle Secrets, never pasted here -- higher HF Hub rate
# limits/download speed, and required if any candidate model is gated.
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = hf_token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token  # some older library versions check this name
    print("HF_TOKEN set from Kaggle secret.")
except Exception as e:
    print(f"No HF_TOKEN secret found ({type(e).__name__}) -- proceeding unauthenticated "
          "(fine for these public models, just slower/rate-limited).")


In [ ]:
# requirements.txt pins peft>=0.19.0, which ships its own Gemma4ClippableLinear
# support -- Gemma 4 no longer needs Unsloth at all (see hf_models.py's
# load_trainable_model() docstring), so there's no separate unsloth install here.
%pip install -q -r requirements.txt
# Kaggle's preinstalled wandb has been observed with its SDK and its own
# bundled generated protobuf file out of sync ("cannot import name 'Imports'
# from wandb.proto.wandb_telemetry_pb2"), which crashes on import -- trl's
# base_trainer.py checks wandb availability at import time, unconditionally,
# regardless of Unsloth. We never use wandb (report_to=[] everywhere) --
# removing it outright is the surest fix, on top of WANDB_DISABLED below.
%pip uninstall -q -y wandb


In [ ]:
import os
# Belt and suspenders alongside hf_models.configure_quiet_logging() (which
# also sets these for every subprocess this notebook launches) -- set here
# too in case anything in the notebook's own kernel process ever imports
# wandb directly.
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


## Run infrastructure: `run_stage()`

Every pipeline stage below goes through this. It never raises -- a stage that OOMs, hits a bug, or has a transient HF Hub hiccup is logged and skipped over, not fatal to the rest of an unattended run. `required_paths` lets a stage declare "don't even try me if this input is missing" (e.g. a GRPO stage needing the preceding SFT stage's checkpoint), so a failure shows up as a clear, attributable skip reason in the log instead of a cryptic error three layers deep in someone else's library. `steps_for_budget()` reads a prior GRPO timing-test's measured per-step cost and sizes the following full run to fit a wall-clock budget automatically, which is what makes "no human between cells" actually work -- without it, the standard advice ("check the timing printout, adjust --max_steps") requires exactly the babysitting this notebook is built to avoid.

In [ ]:
import json
import subprocess
import sys
import time
from pathlib import Path

RESULTS_DIR = Path("./results_check")
STAGE_LOG_PATH = RESULTS_DIR / "stage_log.json"
stage_results = json.loads(STAGE_LOG_PATH.read_text()) if STAGE_LOG_PATH.exists() else []

# Safety-ceiling timeouts (a hard kill-switch if something hangs) -- these are
# NOT the target duration for a stage, just an upper bound. Full GRPO runs
# size their own --max_steps well below their timeout via steps_for_budget().
TIMEOUT_QUICK = 10 * 60        # feasibility check, GRPO timing tests
# This check notebook uses a tiny --n (2) on every benchmark call, so
# even MazeBench's worst-case ~180s/maze (measured on real hardware at 8192
# max_new_tokens, see kaggle_train.ipynb) keeps every eval call well under
# TIMEOUT_EVAL. Not trying to reproduce a meaningful accuracy number here --
# only to prove every stage runs end to end without error.
TIMEOUT_EVAL = 15 * 60         # baseline/checkpoint evals
TIMEOUT_SFT = 20 * 60          # SFT warm-start
TIMEOUT_GRPO_FULL = 20 * 60  # full GRPO training runs, ceiling above their own time budget

# Target wall-clock budget per full GRPO training run -- lower this if your
# weekly quota is tight, raise it if you have room. Applied per condition, so
# the pricier "consistency" condition naturally gets fewer steps than
# "single"/"mixed" for the same budget, without any special-casing.
GRPO_BUDGET_MINUTES = 2


def _save_stage_log():
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    STAGE_LOG_PATH.write_text(json.dumps(stage_results, indent=2))


def run_stage(name, args, required_paths=None, timeout=None):
    """Run `python <args>` as a subprocess. Always returns, never raises."""
    print(f"\n{'='*70}\n\u25b6 {name}\n{'='*70}")
    for p in (required_paths or []):
        if not Path(p).exists():
            print(f"\u23ed  SKIPPED -- required input not found: {p}")
            stage_results.append({"stage": name, "status": "skipped", "reason": f"missing {p}"})
            _save_stage_log()
            return stage_results[-1]

    t0 = time.time()
    try:
        proc = subprocess.run([sys.executable] + args, timeout=timeout)
        elapsed_min = round((time.time() - t0) / 60, 1)
        if proc.returncode == 0:
            print(f"\n\u2705 DONE ({elapsed_min} min)")
            result = {"stage": name, "status": "ok", "elapsed_min": elapsed_min}
        else:
            print(f"\n\u274c FAILED (exit code {proc.returncode}, {elapsed_min} min) -- see output above")
            result = {"stage": name, "status": "failed", "exit_code": proc.returncode, "elapsed_min": elapsed_min}
    except subprocess.TimeoutExpired:
        elapsed_min = round((time.time() - t0) / 60, 1)
        print(f"\n\u23f1  TIMED OUT after {elapsed_min} min (limit {timeout/60:.0f} min)")
        result = {"stage": name, "status": "timeout", "elapsed_min": elapsed_min}
    except Exception as e:
        elapsed_min = round((time.time() - t0) / 60, 1)
        print(f"\n\u274c FAILED (exception: {e})")
        result = {"stage": name, "status": "error", "error": str(e), "elapsed_min": elapsed_min}

    stage_results.append(result)
    _save_stage_log()
    return result


def steps_for_budget(timing_dir, budget_minutes=GRPO_BUDGET_MINUTES, min_steps=3, default_steps=3):
    """Read a prior train_grpo.py run's timing.json and compute a step count
    that fits budget_minutes at that measured per-step cost. Falls back to
    default_steps if timing.json is missing (e.g. the timing test itself
    failed) -- better to attempt a bounded full run than skip it outright."""
    timing_path = Path(timing_dir) / "timing.json"
    if not timing_path.exists():
        print(f"  (no timing.json at {timing_dir}, using default {default_steps} steps)")
        return default_steps
    t = json.loads(timing_path.read_text())
    per_step = t.get("per_step_s", 0)
    if per_step <= 0:
        return default_steps
    steps = max(min_steps, int(budget_minutes * 60 / per_step))
    print(f"  (timing.json: {per_step:.1f}s/step -> {steps} steps fits a {budget_minutes}min budget)")
    return steps


## Feasibility check: does Gemma 4 E2B/E4B LoRA actually fit this GPU?

Don't assume the ~8-10GB (E2B) / ~17GB (E4B) numbers from elsewhere -- confirm on this exact hardware.

In [ ]:
run_stage("Feasibility check (all candidate models)", ["check_finetune_feasibility.py"], timeout=TIMEOUT_QUICK)


## Phase 1: baselines -- Gemma 4 E2B/E4B + AlphaMaze on MazeBench and GridRoute 5x5

AlphaMaze should land near its published 93% on MazeBench (using their real scoring code via the submodule) -- this is the harness sanity check. Gemma 4's numbers on both benchmarks are the actual open question: no SLM has been tested on GridRoute anywhere in the literature found so far.

(This notebook uses n=2 everywhere below -- too few samples for the 93% figure to mean anything statistically. The point here is only proving the harness runs; the real replication check lives in `kaggle_train.ipynb`.)

In [ ]:
ALPHAMAZE_LOCAL_PATH = "data/models/alphamaze-v0.2-1.5b"
os.makedirs("data/models", exist_ok=True)
try:
    if not os.path.isdir(ALPHAMAZE_LOCAL_PATH):
        from huggingface_hub import snapshot_download
        snapshot_download("homebrewltd/AlphaMaze-v0.2-1.5B", local_dir=ALPHAMAZE_LOCAL_PATH)
    print("AlphaMaze checkpoint ready.")
except Exception as e:
    print(f"\u26a0\ufe0f  AlphaMaze checkpoint download failed ({e}) -- AlphaMaze-specific "
          "stages below will be skipped (they require-path-check this directory).")


This cell exercises the exact same AlphaMaze-on-MazeBench path as `kaggle_train.ipynb`'s Phase 1 replication check (full precision, real submodule scoring via `alphamaze_reference`) but with only 2 mazes and the standard 4096-token floor instead of 16384 -- not trying to reproduce the published ~93% here, only to prove the checkpoint loads, generates, streams, and scores correctly end to end. The real replication check lives in `kaggle_train.ipynb`.

In [ ]:
run_stage("AlphaMaze baseline: MazeBench (full precision, pipeline check)",
          ["eval.py", "--model", "alphamaze", "--benchmark", "mazebench", "--n", "2",
           "--no_4bit", "--output_dir", "./results_check/eval"],
          required_paths=[ALPHAMAZE_LOCAL_PATH], timeout=15 * 60)
run_stage("AlphaMaze baseline: GridRoute NL",
          ["eval.py", "--model", "alphamaze", "--benchmark", "gridroute-nl", "--grid_size", "5",
           "--n", "2", "--output_dir", "./results_check/eval"],
          required_paths=[ALPHAMAZE_LOCAL_PATH], timeout=TIMEOUT_EVAL)


In [ ]:
run_stage("Gemma 4 E2B baseline: MazeBench",
          ["eval.py", "--model", "gemma4-e2b", "--benchmark", "mazebench", "--n", "2",
           "--max_new_tokens", "1024", "--output_dir", "./results_check/eval"], timeout=TIMEOUT_EVAL)
run_stage("Gemma 4 E2B baseline: GridRoute NL",
          ["eval.py", "--model", "gemma4-e2b", "--benchmark", "gridroute-nl", "--grid_size", "5",
           "--n", "2", "--output_dir", "./results_check/eval"], timeout=TIMEOUT_EVAL)


E4B baseline (inference-only, no LoRA) is much lighter than E4B *training* -- run this regardless of what the feasibility check said about training, since it doesn't need LoRA attachment at all, just enough VRAM to hold the weights at 4-bit.

In [ ]:
run_stage("Gemma 4 E4B baseline: MazeBench",
          ["eval.py", "--model", "gemma4-e4b", "--benchmark", "mazebench", "--n", "2",
           "--max_new_tokens", "1024", "--output_dir", "./results_check/eval"], timeout=TIMEOUT_EVAL)
run_stage("Gemma 4 E4B baseline: GridRoute NL",
          ["eval.py", "--model", "gemma4-e4b", "--benchmark", "gridroute-nl", "--grid_size", "5",
           "--n", "2", "--output_dir", "./results_check/eval"], timeout=TIMEOUT_EVAL)


## Phase 1b: full-precision ablation for Gemma 4 (how much does 4-bit cost?)

Gemma 4's baselines above ran 4-bit-quantized -- necessary for *training* (LoRA/GRPO need the VRAM headroom), but baseline inference has no such requirement, so it's worth checking directly rather than assuming the gap is small. Same `--n`/seed as the corresponding 4-bit cell above, so this is a paired comparison on identical tasks, not just a same-average different-sample one. `eval.py` records `load_in_4bit` in its output JSON specifically so the aggregation table below can show both side by side. Gemma 4 E4B at full precision is the one most likely to not fit the T4 at all (4-bit exists exactly to avoid this) -- if that cell fails, that itself is a useful, reportable data point, not a notebook bug.

AlphaMaze isn't part of this ablation at all: it always runs full precision, enforced directly in `eval.py` regardless of any `--load_in_4bit` flag, not just by convention in this notebook. It's the one fixed reference point every other number here gets compared against and it's never trained further in Phase 2, so there's no training-time VRAM constraint forcing a quantization tradeoff the way there is for Gemma 4 -- no reason to ever accept the noise 4-bit would add to its numbers.

In [ ]:
run_stage("[full-precision] Gemma 4 E2B baseline: MazeBench",
          ["eval.py", "--model", "gemma4-e2b", "--benchmark", "mazebench", "--n", "2",
           "--max_new_tokens", "1024", "--no_4bit", "--output_dir", "./results_check/eval"], timeout=TIMEOUT_EVAL)
run_stage("[full-precision] Gemma 4 E2B baseline: GridRoute NL",
          ["eval.py", "--model", "gemma4-e2b", "--benchmark", "gridroute-nl", "--grid_size", "5",
           "--n", "2", "--no_4bit", "--output_dir", "./results_check/eval"], timeout=TIMEOUT_EVAL)


In [ ]:
run_stage("[full-precision] Gemma 4 E4B baseline: MazeBench",
          ["eval.py", "--model", "gemma4-e4b", "--benchmark", "mazebench", "--n", "2",
           "--max_new_tokens", "1024", "--no_4bit", "--output_dir", "./results_check/eval"], timeout=TIMEOUT_EVAL)
run_stage("[full-precision] Gemma 4 E4B baseline: GridRoute NL",
          ["eval.py", "--model", "gemma4-e4b", "--benchmark", "gridroute-nl", "--grid_size", "5",
           "--n", "2", "--no_4bit", "--output_dir", "./results_check/eval"], timeout=TIMEOUT_EVAL)


## Phase 2, recipe 1: single-format (GridRoute NL only) on Gemma 4 E2B

SFT warm-start, a short GRPO timing test, then a full run sized to fit `GRPO_BUDGET_MINUTES` at the timing test's measured per-step cost -- see the run-infrastructure cell above for why this is computed rather than manually tuned.

In [ ]:
SFT_SINGLE_DIR = "./results_check/sft_gemma4-e2b_single"
run_stage("SFT: single-format (Gemma 4 E2B)",
          ["train_sft.py", "--model", "gemma4-e2b", "--format", "nl", "--grid_size", "5",
           "--n_tasks", "16", "--epochs", "1", "--output_dir", SFT_SINGLE_DIR], timeout=TIMEOUT_SFT)


In [ ]:
GRPO_SINGLE_TIMING_DIR = "./results_check/grpo_gemma4-e2b_single_timing"
run_stage("GRPO timing test: single-format",
          ["train_grpo.py", "--model", "gemma4-e2b", "--adapter_path", SFT_SINGLE_DIR,
           "--condition", "single", "--grid_size", "5", "--n_tasks", "8", "--max_steps", "2", "--max_completion_length", "1024",
           "--output_dir", GRPO_SINGLE_TIMING_DIR],
          required_paths=[SFT_SINGLE_DIR], timeout=TIMEOUT_QUICK)


In [ ]:
GRPO_SINGLE_DIR = "./results_check/grpo_gemma4-e2b_single"
single_steps = steps_for_budget(GRPO_SINGLE_TIMING_DIR)
run_stage(f"GRPO: single-format full run ({single_steps} steps)",
          ["train_grpo.py", "--model", "gemma4-e2b", "--adapter_path", SFT_SINGLE_DIR,
           "--condition", "single", "--grid_size", "5", "--n_tasks", "8",
           "--max_steps", str(single_steps), "--max_completion_length", "1024", "--output_dir", GRPO_SINGLE_DIR],
          required_paths=[SFT_SINGLE_DIR], timeout=TIMEOUT_GRPO_FULL)


In [ ]:
run_stage("Eval single-format checkpoint: MazeBench",
          ["eval.py", "--model", "gemma4-e2b", "--checkpoint", GRPO_SINGLE_DIR,
           "--benchmark", "mazebench", "--n", "2", "--max_new_tokens", "1024",
           "--output_dir", "./results_check/eval"],
          required_paths=[GRPO_SINGLE_DIR], timeout=TIMEOUT_EVAL)
run_stage("Eval single-format checkpoint: GridRoute NL",
          ["eval.py", "--model", "gemma4-e2b", "--checkpoint", GRPO_SINGLE_DIR,
           "--benchmark", "gridroute-nl", "--grid_size", "5", "--n", "2", "--output_dir", "./results_check/eval"],
          required_paths=[GRPO_SINGLE_DIR], timeout=TIMEOUT_EVAL)


## Phase 2, recipe 2: mixed-format (NL + token, naive)

Does training on both formats (interleaved, same underlying grids) do better than single-format alone -- for either benchmark?

In [ ]:
SFT_MIXED_DIR = "./results_check/sft_gemma4-e2b_mixed"
run_stage("SFT: mixed-format (Gemma 4 E2B)",
          ["train_sft.py", "--model", "gemma4-e2b", "--format", "mixed", "--grid_size", "5",
           "--n_tasks", "16", "--epochs", "1", "--output_dir", SFT_MIXED_DIR], timeout=TIMEOUT_SFT)


In [ ]:
GRPO_MIXED_TIMING_DIR = "./results_check/grpo_gemma4-e2b_mixed_timing"
run_stage("GRPO timing test: mixed-format",
          ["train_grpo.py", "--model", "gemma4-e2b", "--adapter_path", SFT_MIXED_DIR,
           "--condition", "mixed", "--grid_size", "5", "--n_tasks", "8", "--max_steps", "2", "--max_completion_length", "1024",
           "--output_dir", GRPO_MIXED_TIMING_DIR],
          required_paths=[SFT_MIXED_DIR], timeout=TIMEOUT_QUICK)


In [ ]:
GRPO_MIXED_DIR = "./results_check/grpo_gemma4-e2b_mixed"
mixed_steps = steps_for_budget(GRPO_MIXED_TIMING_DIR)
run_stage(f"GRPO: mixed-format full run ({mixed_steps} steps)",
          ["train_grpo.py", "--model", "gemma4-e2b", "--adapter_path", SFT_MIXED_DIR,
           "--condition", "mixed", "--grid_size", "5", "--n_tasks", "8",
           "--max_steps", str(mixed_steps), "--max_completion_length", "1024", "--output_dir", GRPO_MIXED_DIR],
          required_paths=[SFT_MIXED_DIR], timeout=TIMEOUT_GRPO_FULL)


In [ ]:
run_stage("Eval mixed-format checkpoint: MazeBench",
          ["eval.py", "--model", "gemma4-e2b", "--checkpoint", GRPO_MIXED_DIR,
           "--benchmark", "mazebench", "--n", "2", "--max_new_tokens", "1024",
           "--output_dir", "./results_check/eval"],
          required_paths=[GRPO_MIXED_DIR], timeout=TIMEOUT_EVAL)
run_stage("Eval mixed-format checkpoint: GridRoute NL",
          ["eval.py", "--model", "gemma4-e2b", "--checkpoint", GRPO_MIXED_DIR,
           "--benchmark", "gridroute-nl", "--grid_size", "5", "--n", "2", "--output_dir", "./results_check/eval"],
          required_paths=[GRPO_MIXED_DIR], timeout=TIMEOUT_EVAL)


## Phase 2, recipe 3: consistency-reward (one candidate recipe, not this project's headline claim)

Adapts Elhady et al.'s cross-lingual consistency-reward mechanism to cross-format spatial reasoning -- try it, report honestly whether it beats `mixed` or not. See `train_grpo.py`'s `make_reward_fn` docstring for the exact mechanism. This condition generates an extra partner completion per reward call (roughly double the per-step cost of single/mixed) -- `steps_for_budget()` accounts for that automatically since it reads THIS condition's own timing test, not single/mixed's.

In [ ]:
SFT_CONSISTENCY_DIR = SFT_MIXED_DIR  # consistency reuses the same mixed-format SFT warm-start
GRPO_CONSISTENCY_TIMING_DIR = "./results_check/grpo_gemma4-e2b_consistency_timing"
run_stage("GRPO timing test: consistency-reward",
          ["train_grpo.py", "--model", "gemma4-e2b", "--adapter_path", SFT_CONSISTENCY_DIR,
           "--condition", "consistency", "--grid_size", "5", "--n_tasks", "8", "--max_steps", "2", "--max_completion_length", "1024",
           "--output_dir", GRPO_CONSISTENCY_TIMING_DIR],
          required_paths=[SFT_CONSISTENCY_DIR], timeout=TIMEOUT_QUICK)


In [ ]:
GRPO_CONSISTENCY_DIR = "./results_check/grpo_gemma4-e2b_consistency"
consistency_steps = steps_for_budget(GRPO_CONSISTENCY_TIMING_DIR)
run_stage(f"GRPO: consistency-reward full run ({consistency_steps} steps)",
          ["train_grpo.py", "--model", "gemma4-e2b", "--adapter_path", SFT_CONSISTENCY_DIR,
           "--condition", "consistency", "--grid_size", "5", "--n_tasks", "8",
           "--max_steps", str(consistency_steps), "--max_completion_length", "1024", "--output_dir", GRPO_CONSISTENCY_DIR],
          required_paths=[SFT_CONSISTENCY_DIR], timeout=TIMEOUT_GRPO_FULL)


In [ ]:
run_stage("Eval consistency-reward checkpoint: MazeBench",
          ["eval.py", "--model", "gemma4-e2b", "--checkpoint", GRPO_CONSISTENCY_DIR,
           "--benchmark", "mazebench", "--n", "2", "--max_new_tokens", "1024",
           "--output_dir", "./results_check/eval"],
          required_paths=[GRPO_CONSISTENCY_DIR], timeout=TIMEOUT_EVAL)
run_stage("Eval consistency-reward checkpoint: GridRoute NL",
          ["eval.py", "--model", "gemma4-e2b", "--checkpoint", GRPO_CONSISTENCY_DIR,
           "--benchmark", "gridroute-nl", "--grid_size", "5", "--n", "2", "--output_dir", "./results_check/eval"],
          required_paths=[GRPO_CONSISTENCY_DIR], timeout=TIMEOUT_EVAL)


## Run summary

At-a-glance status of every stage attempted -- read this first when checking back on an unattended run, before digging into the full cell outputs above.

In [ ]:
import pandas as pd

summary_df = pd.DataFrame(stage_results)
if not summary_df.empty:
    ok = (summary_df["status"] == "ok").sum()
    print(f"{ok}/{len(summary_df)} stages completed successfully.\n")
pd.set_option("display.max_colwidth", None)
summary_df


## Pipeline check verdict

`run_stage()` deliberately never raises -- that's what lets an unattended multi-hour real run survive one bad stage. But this notebook exists to answer one question (did everything work?), so this cell inverts that: it fails loudly, on purpose, the moment anything didn't complete cleanly.

In [ ]:
failed = [r for r in stage_results if r["status"] != "ok"]

print(f"\n{'='*70}")
if failed:
    print(f"CHECK FAILED -- {len(failed)}/{len(stage_results)} stage(s) did not complete cleanly:\n")
    for r in failed:
        detail = r.get("error") or (f"exit code {r['exit_code']}" if "exit_code" in r else r.get("reason", ""))
        print(f"  [{r['status'].upper()}] {r['stage']}" + (f" -- {detail}" if detail else ""))
    print(f"{'='*70}")
    raise RuntimeError(
        f"{len(failed)} pipeline stage(s) failed or were skipped -- see the list above and the "
        "full cell output higher in this notebook for the actual error/traceback. Fix before "
        "trusting a real kaggle_train.ipynb run."
    )
else:
    print(f"CHECK PASSED -- all {len(stage_results)} attempted stages completed successfully.")
    print("Every stage of the real pipeline (setup, feasibility, Phase 1, Phase 1b, "
          "Phase 2 recipes 1-3, eval) ran end to end on this hardware without error.")
    print(f"{'='*70}")


## Aggregate benchmark results into one comparison table

In [ ]:
import glob

rows = []
for path in sorted(glob.glob("./results_check/eval/*.json")):
    try:
        with open(path) as f:
            d = json.load(f)
    except (json.JSONDecodeError, OSError) as e:
        print(f"Skipping unreadable results file {path}: {e}")
        continue
    label = path.split("/")[-1]
    row = {"file": label, "model": d.get("model"), "checkpoint": d.get("checkpoint") or "(none)",
           "benchmark": d.get("benchmark"), "n": d.get("n"),
           "4bit": d.get("load_in_4bit")}  # AlphaMaze rows are always False (enforced in eval.py);
                                            # False Gemma 4 rows are the Phase 1b ablation
    if "mazebench" in d.get("benchmark", ""):
        row["score"] = d.get("accuracy")
        row["used_official_scoring"] = d.get("used_official_scoring")
    else:
        row["valid_rate"] = d.get("valid_rate")
        row["optimal_rate"] = d.get("optimal_rate")
    rows.append(row)

results_df = pd.DataFrame(rows)
if not results_df.empty:
    results_df.to_csv("./results_check/comparison_table.csv", index=False)
else:
    print("No eval result files found yet in ./results_check/eval/ -- nothing to aggregate.")
results_df


## Download results

Small, diagnostic-only -- this is the check run's tiny-sample output, not a substitute for `kaggle_train.ipynb`'s real results.

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/results_check", "zip", "./results_check")
print("Saved: /kaggle/working/results_check.zip -- download it from the Kaggle output panel on the right.")
print("Includes stage_log.json (this run's full stage-by-stage status) and comparison_table.csv.")
